<a href="https://colab.research.google.com/github/sotesh1516/Transformer-Attention_Is_All_You_Need/blob/main/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import math
import torch

The following are part of tokenization, converting text into discreet IDs.

In [2]:
def build_token_to_id_vocab(sentences, specials=('<pad>', '<bos>', '<eos>', '<unk>')):
    # build a token-to-id dict with specials first, then corpus tokens in first-seen order.
    vocab_dict = {}
    global_id_counter = 0
    for sp in specials:
        vocab_dict[sp] = global_id_counter
        global_id_counter+=1
    for w in sentences:
        w_arr = w.split()
        for t in w_arr:
            if t not in vocab_dict:
                vocab_dict[t] = global_id_counter
                global_id_counter+=1

    return vocab_dict

def build_id_to_token_vocab(token_to_id):
    # build the inverse id-to-token dictionary from token_to_id
    id_to_token = {}
    for token in token_to_id:
        id_to_token[token_to_id[token]] = token

    return id_to_token

def encode_sentence_to_ids(sentence, token_to_id, unk_token='<unk>'):
    # convert whitespace tokens of `sentence` to ids via `token_to_id`, using `unk_token`'s id for OOV
    int_token_id = []
    whitespace_token = sentence.split()
    for t in whitespace_token:
        if t not in token_to_id:
            int_token_id.append(token_to_id[unk_token])
        else:
            int_token_id.append(token_to_id[t])

    return int_token_id

def decode_ids_to_tokens(ids, id_to_token):
    # map each id in ids to its token string via id_to_token and return the list
    tokens = []
    for id in ids:
        tokens.append(id_to_token[id])
    return tokens

def pad_id_sequence(ids, max_len, pad_id):
    # return a list of length exactly max_len, padding with pad_id or truncating.
    if max_len > len(ids):
        return ids + [pad_id for i in range(max_len - len(ids))]
    return ids[:max_len]

def stack_padded_sequences_to_batch(padded_sequences):
    """Stack a list of equal-length padded id sequences into a 2D LongTensor batch."""
    # stack padded id sequences into a (B, L) torch.long tensor
    return torch.tensor(padded_sequences).long()

Embeddings and Positional Encoding

In [3]:
def scale_embeddings_by_sqrt_d_model(embeddings, d_model):
    """Scale a token embedding tensor by sqrt(d_model)."""
    # rescale embeddings by sqrt(d_model) as in the original Transformer paper
    return embeddings * math.sqrt(d_model)

def compute_positional_div_term(d_model):
    """
    Note:
    - Using several different frequencies gives each position a distinguishable vector pattern.
    - Assign each feature channel(dimension) its own angular frequency
    - Here we are producing pair-wise freq
        - frequency for pair 0 (dim 0 and 1)
        - frequency for pair 1
        - frequency for pair 2
    - Finally we can do pos * 1000^(-2i/d_model) -> this is the current step
    """
    # return a 1D FloatTensor of length d_model // 2 holding the sinusoidal frequency divisors
    output_len = d_model // 2
    i = torch.arange(output_len, dtype=torch.float32)
    freq = 10000 ** (-2.0 * i / d_model)
    return freq

def build_position_index_column(max_len):
    """
    Return a (max_len, 1) float tensor of [0, 1, ..., max_len-1].

    The sinusoidal positional encoding evaluates sine and cosine at every (position, frequency) pair.
    """
    # build a column vector of position indices from 0 to max_len-1
    pos_indices = torch.empty(max_len,1)
    for i in range(max_len):
        pos_indices[i] = torch.tensor(i).float()
    return pos_indices

def fill_even_indices_with_sin(pe, position, div_term):
    """Fill even feature indices of pe with sin(position * div_term)."""
    # write sin(position * div_term) into the even-indexed columns of pe and return it
    for i in range(len(pe)):
        for j in range(len(pe[0])):
            if j % 2 == 0: #even dimension
                current_dim_freq = j // 2
                pe[i, j] = torch.sin(position[i] * div_term[current_dim_freq])

    return pe

def fill_odd_indices_with_cos(pe, position, div_term):
    # fill the odd-indexed columns of pe with cos(position * div_term)
    for i in range(len(pe)):
        for j in range(len(pe[0])):
            if j % 2 != 0: #odd dimension
                current_dim_freq = j // 2
                pe[i, j] = torch.cos(position[i] * div_term[current_dim_freq])

    return pe

def build_sinusoidal_positional_encoding(max_len, d_model):
    """Assemble the (max_len, d_model) sinusoidal positional encoding matrix."""
    # build the (max_len, d_model) sinusoidal positional encoding matrix

    positional_encoding = torch.zeros(max_len, d_model)
    positional_index_cols = build_position_index_column(max_len) # max_len x 1
    divisor_terms = compute_positional_div_term(d_model)
    even_filled_positional_encoding = fill_even_indices_with_sin(positional_encoding, positional_index_cols, divisor_terms)
    positional_encoding = fill_odd_indices_with_cos(even_filled_positional_encoding, positional_index_cols, divisor_terms)
    return positional_encoding

def add_positional_encoding_to_embeddings(embedded_batch, positional_encoding):
    # add the first L rows of positional_encoding to embedded_batch and return the sum.
    # for each batch, position resets
    input_batch = embedded_batch.shape[0]
    num_input_token = embedded_batch.shape[1]
    d_model = embedded_batch.shape[2] #can get this from the pos encoding too
    positional_embedding = torch.empty(input_batch, num_input_token, d_model)
    for i in range(input_batch):
        positional_embedding[i] = embedded_batch[i] + positional_encoding[:num_input_token, :]

    return positional_embedding

Masks and Scaled Dot-Product Attention

In [28]:
def build_padding_mask(token_ids, pad_id):
    """Return a (B, 1, 1, L) bool mask: True where token_ids != pad_id.
    - multiple batch
        - 1 Head
            - 1 query
                - multiple keys
    """
    # build a boolean mask marking non-pad positions, shaped for broadcasting against attention scores
    num_batches = token_ids.shape[0]
    num_keys = token_ids.shape[1]
    bool_mask = torch.empty((num_batches, 1, 1, num_keys), dtype=torch.bool)

    for b in range(num_batches):
        for key in range(num_keys):
            bool_mask[b, 0, 0, key] = token_ids[b, key] != pad_id

    return bool_mask

def build_causal_mask(seq_len):
    """Return a (1, 1, seq_len, seq_len) bool mask, True on and below diagonal."""
    # build a lower-triangular boolean causal mask of shape (1, 1, seq_len, seq_len)
    lower_triangular_bool_mask = torch.tril(
    torch.ones(seq_len, seq_len, dtype=torch.bool)
    )

    lower_triangular_bool_mask = lower_triangular_bool_mask.unsqueeze(0).unsqueeze(0)

    return lower_triangular_bool_mask

def combine_padding_and_causal_masks(padding_mask, causal_mask):
    # combine a (B,1,1,L) padding mask with a (1,1,L,L) causal mask into (B,1,L,L).
    # broadcasting works right to left, using a simple rule
      # same size OR one of them is 1
    return causal_mask & padding_mask

def compute_raw_attention_scores(query, key):
    """Compute raw attention scores Q @ K^T over the last two dimensions."""
    # matmul query with the transpose of key over the last two axes
    # 2 x 2 -> query
    # 3 x 2 -> key
    return query @ key.transpose(-2,-1)

def scale_attention_scores(scores, d_k):
    # divide raw attention scores by sqrt(d_k) to stabilize softmax inputs
    return scores / math.sqrt(d_k)

def mask_attention_scores_with_neg_inf(scores, mask):
    """Set entries of scores where mask is False to -inf."""
    # replace blocked positions of scores with negative infinity
    # masked_fill replaces positions where its mask is True, hence ~
    return scores.masked_fill(~mask, float("-inf"))

def softmax_attention_weights(masked_scores):
    # softmax over the last axis, zeroing rows that are entirely -inf
    init_prob =  torch.softmax(masked_scores, dim=-1)
    row_mask = torch.isneginf(masked_scores).all(dim=-1, keepdim=True)
    return torch.where(row_mask, 0.0, init_prob)

def apply_attention_weights_to_values(attention_weights, value):
    """Multiply attention weights by the value matrix to produce context vectors."""
    # combine attention weights (..., Lq, Lk) with value (..., Lk, d_v)
    return attention_weights @ value

def scaled_dot_product_attention(query, key, value, mask=None):
    """Run scaled dot-product attention; return (context, attention_weights)."""
    # chain raw scores, scale by sqrt(d_k), optionally mask, softmax, then mix values
    raw_weighted_score = compute_raw_attention_scores(query, key)
    scaled_attention_score = scale_attention_scores(raw_weighted_score, key.shape[-1])
    if mask != None:
        masked_attention_score = mask_attention_scores_with_neg_inf(scaled_attention_score, mask)
    else:
        masked_attention_score = scaled_attention_score

    softmax_attention_score = softmax_attention_weights(masked_attention_score)
    context_vector = apply_attention_weights_to_values(softmax_attention_score, value)
    return (context_vector, softmax_attention_score)

Multi-head Attention

In [5]:
def split_last_dim_into_heads(tensor, num_heads):
    # reshape (B, L, d_model) into (B, L, num_heads, d_model // num_heads)
    batch, length, d_model = tensor.shape
    return tensor.reshape(batch, length, num_heads, d_model // num_heads)

def transpose_heads_before_sequence(split_tensor):
    # rearrange (B, L, num_heads, d_k) into (B, num_heads, L, d_k).
    return split_tensor.transpose(1,2)

def transpose_heads_before_sequence(split_tensor):
    # rearrange (B, L, num_heads, d_k) into (B, num_heads, L, d_k).
    # one attention-head at a time, instead of one token in each attention-head
    return split_tensor.transpose(1,2)

def merge_heads_back_to_model_dim(multi_head_tensor):
    # merge the head axis back into the feature axis to reconstruct d_model
    # switch num_heads and length first
    batch, num_heads, length, d_k = multi_head_tensor.shape
    return multi_head_tensor.transpose(1,2).reshape(batch, length, num_heads * d_k)

def apply_linear_projection(x, weight, bias):
    # return x @ weight^T + bias (bias may be None) with shape (..., out_features)
    # (batch, length, in_features) x (out_features, in_features).T
        # broadcasting is used for earlier dim (batch, length)
    return x @ weight.T + bias if bias is not None else x @ weight.T

def project_to_query_key_value(x, w_q, b_q, w_k, b_k, w_v, b_v):
    # project x into separate query, key, and value tensors via three linear layers
    query = apply_linear_projection(x, w_q, b_q)
    key = apply_linear_projection(x, w_k, b_k)
    value = apply_linear_projection(x, w_v, b_v)

    return query, key, value

def split_qkv_into_heads(q, k, v, num_heads):
    # split each of q, k, v into (B, num_heads, L, d_k) and return as a tuple
    q_reshaped_transposed = transpose_heads_before_sequence(split_last_dim_into_heads(q, num_heads))
    k_reshaped_transposed = transpose_heads_before_sequence(split_last_dim_into_heads(k, num_heads))
    v_reshaped_transposed = transpose_heads_before_sequence(split_last_dim_into_heads(v, num_heads))

    return q_reshaped_transposed, k_reshaped_transposed, v_reshaped_transposed

def multi_head_scaled_dot_product_attention(q_h, k_h, v_h, mask=None):
    # run scaled dot-product attention over per-head Q, K, V and return (context, weights)
    context_vector, softmax_attention_score = scaled_dot_product_attention(q_h, k_h, v_h, mask)
    return context_vector, softmax_attention_score

def merge_heads_and_project_output(context, w_o, b_o):
    # merge the head axis back into d_model and apply the output linear projection.
    single_head_context_tensor = merge_heads_back_to_model_dim(context)
    linear_output = apply_linear_projection(single_head_context_tensor, w_o, b_o)
    return linear_output

def assemble_multi_head_attention_forward(query, key, value, w_q, w_k, w_v, w_o, num_heads, mask=None):
    # project Q/K/V, split into heads, run scaled dot-product attention, merge heads, output projection.
    query_q, key_q, value_q = project_to_query_key_value(query, w_q, None, w_k, None, w_v, None)
    query_k, key_k, value_k = project_to_query_key_value(key, w_q, None, w_k, None, w_v, None)
    query_v, key_v, value_v = project_to_query_key_value(value, w_q, None, w_k, None, w_v, None)

    query, key, value = split_qkv_into_heads(query_q, key_k, value_v, num_heads)
    context_vector, softmax_attention_score = multi_head_scaled_dot_product_attention(query, key, value, mask)
    single_head_context_tensor = merge_heads_and_project_output(context_vector, w_o, None)

    return single_head_context_tensor

Feed-forward, Layernorm, and Dropout

In [6]:
def apply_ffn_first_linear_and_relu(x, w1, b1):
    # project x by w1, add b1, then apply a ReLU activation.
    # same w1 and b1 for each batch
    return torch.relu(x @ w1 + b1)

def apply_ffn_second_linear(hidden, w2, b2):
    # project hidden (..., d_ff) back to (..., d_model) via w2 and b2.
    return hidden @ w2 + b2

def position_wise_feed_forward_network(x, w1, b1, w2, b2):
    # compose the two FFN linears with a ReLU in between, returning shape (B, T, d_model).
    hidden = apply_ffn_first_linear_and_relu(x, w1, b1)
    ffn_output = apply_ffn_second_linear(hidden, w2, b2)

    return ffn_output

def compute_layer_norm_mean_and_variance(x):
    # return (mean, variance) reduced over the last dim with shape (..., 1)
    # correction=0 calc pop var, dim=-1 works on the last dim, keepdim keeps (...,1)
    mean = torch.mean(x, dim=-1, keepdim=True)
    variance = torch.var(x, correction=0, dim=-1, keepdim=True)
    return mean, variance

def normalize_and_scale_with_gamma_beta(x, gamma, beta, eps=1e-5):
    # standardize x along the last axis then apply gamma and beta affine transform
    # gamma for scaling and beta for shift
    # applying these params is a way to allow flexibility in learning, since layernorm can be restrictive
    mean, var = compute_layer_norm_mean_and_variance(x)
    standardized_ouput = (x - mean) / torch.sqrt(var + eps)
    normalized_ouput = gamma * standardized_ouput + beta
    return normalized_ouput

def apply_residual_add_and_norm(residual_input, sublayer_output, gamma, beta, eps=1e-5):
    # combine the residual with the sublayer output and layer-normalize the result.
    added_residual = residual_input + sublayer_output
    normalized_ouput = normalize_and_scale_with_gamma_beta(added_residual, gamma, beta, eps)
    return normalized_ouput

def apply_dropout_with_keep_mask(x, keep_mask, keep_prob):
    # multiply x by the boolean keep_mask and rescale by 1/keep_prob.
    return (x * keep_mask.float()) / keep_prob

Encoder, Decoder, and Full Model

In [7]:
def encoder_layer_self_attention_sublayer(x, w_q, w_k, w_v, w_o, gamma, beta, num_heads, src_mask):
    # run multi-head self-attention on x and wrap with residual add-and-norm.
    single_head_context_vector = assemble_multi_head_attention_forward(x, x, x, w_q, w_k, w_v, w_o, num_heads, src_mask)
    normalized_output = apply_residual_add_and_norm(x, single_head_context_vector, gamma, beta)
    return normalized_output

def encoder_layer_feed_forward_sublayer(x, w1, b1, w2, b2, gamma, beta):
    # run the position-wise FFN on x and wrap it with residual add-and-norm.
    ffn_output = position_wise_feed_forward_network(x, w1, b1, w2, b2)
    normalized_output = apply_residual_add_and_norm(x, ffn_output, gamma, beta)
    return normalized_output

def assemble_encoder_layer(x, layer_params, num_heads, src_mask):
    # chain the self-attention sublayer and the feed-forward sublayer using layer_params.
    self_attention_sublayer = encoder_layer_self_attention_sublayer(x, layer_params['w_q'], layer_params['w_k'],
    layer_params['w_v'], layer_params['w_o'], layer_params['attn_gamma'], layer_params['attn_beta'], num_heads, src_mask)
    feed_forward_sublayer = encoder_layer_feed_forward_sublayer(self_attention_sublayer, layer_params['w1'],
    layer_params['b1'], layer_params['w2'], layer_params['b2'], layer_params['ffn_gamma'], layer_params['ffn_beta'])
    return feed_forward_sublayer

def stack_encoder_layers(x, encoder_layer_params_list, num_heads, src_mask):
    # sequentially apply each encoder layer to the running hidden state and return the final tensor.
    if len(encoder_layer_params_list) == 0:
        return x

    encoder_layer = assemble_encoder_layer(x, encoder_layer_params_list[0], num_heads, src_mask)

    if len(encoder_layer_params_list) == 1:
        return encoder_layer

    for i in range(1, len(encoder_layer_params_list)):
        encoder_layer = assemble_encoder_layer(encoder_layer, encoder_layer_params_list[i], num_heads, src_mask)

    return encoder_layer

def decoder_layer_masked_self_attention_sublayer(y, w_q, w_k, w_v, w_o, gamma, beta, num_heads, tgt_mask):
    # run masked multi-head self-attention on y and wrap with residual add-and-norm.
    single_head_context_vector = assemble_multi_head_attention_forward(y, y, y, w_q, w_k, w_v, w_o, num_heads, tgt_mask)
    normalized_output = apply_residual_add_and_norm(y, single_head_context_vector, gamma, beta)
    return normalized_output

def decoder_layer_cross_attention_sublayer(y, encoder_output, w_q, w_k, w_v, w_o, gamma, beta, num_heads, src_mask):
    # run multi-head cross-attention (Q from y, K/V from encoder_output) and wrap with add-and-norm
    # asymmetric tgt(decoder) and src(encoder) seq length can mess up the masking process, so need to reshape the mask
        # wont face this problem in encoder since there is cross-attention(mixing of seq length)
    if y.shape[-2] != encoder_output.shape[-2] and src_mask is not None:
        src_mask = src_mask.unsqueeze(1).unsqueeze(1)
    single_head_context_vector = assemble_multi_head_attention_forward(y, encoder_output, encoder_output, w_q, w_k, w_v, w_o, num_heads, src_mask)
    normalized_output = apply_residual_add_and_norm(y, single_head_context_vector, gamma, beta)
    return normalized_output

def decoder_layer_feed_forward_sublayer(y, w1, b1, w2, b2, gamma, beta):
    # run the position-wise FFN on y and wrap it with residual add-and-norm
    ffn_output = position_wise_feed_forward_network(y, w1, b1, w2, b2)
    normalized_output = apply_residual_add_and_norm(y, ffn_output, gamma, beta)
    return normalized_output

def assemble_decoder_layer(y, encoder_output, layer_params, num_heads, src_mask, tgt_mask):
    """Run a full decoder layer: masked self-attention, cross-attention, then FFN."""
    # chain the three decoder sublayers using params from layer_params.
    masked_self_attention_sublayer = decoder_layer_masked_self_attention_sublayer(y, layer_params['w_q_self'], layer_params['w_k_self'],
    layer_params['w_v_self'], layer_params['w_o_self'], layer_params['self_gamma'], layer_params['self_beta'], num_heads, tgt_mask)
    self_attention_sublayer = decoder_layer_cross_attention_sublayer(masked_self_attention_sublayer, encoder_output, layer_params['w_q_cross'], layer_params['w_k_cross'],
    layer_params['w_v_cross'], layer_params['w_o_cross'], layer_params['cross_gamma'], layer_params['cross_beta'], num_heads, src_mask)
    feed_forward_sublayer = encoder_layer_feed_forward_sublayer(self_attention_sublayer, layer_params['w1'],
    layer_params['b1'], layer_params['w2'], layer_params['b2'], layer_params['ffn_gamma'], layer_params['ffn_beta'])
    return feed_forward_sublayer

def stack_decoder_layers(y, encoder_output, decoder_layer_params_list, num_heads, src_mask, tgt_mask):
    # sequentially apply each decoder layer to the running target hidden state.
    if len(decoder_layer_params_list) == 0:
        return y

    decoder_layer = assemble_decoder_layer(y, encoder_output, decoder_layer_params_list[0], num_heads, src_mask, tgt_mask)

    if len(decoder_layer_params_list) == 1:
        return decoder_layer

    for i in range(1, len(decoder_layer_params_list)):
        decoder_layer = assemble_decoder_layer(decoder_layer, encoder_output, decoder_layer_params_list[i], num_heads, src_mask, tgt_mask)

    return decoder_layer

def apply_final_output_projection(decoder_output, output_projection_weight, output_projection_bias=None):
    # project decoder hidden states (B, T, D) to vocabulary logits (B, T, V).
    linear_layer = apply_linear_projection(decoder_output, output_projection_weight, output_projection_bias)
    return linear_layer

def tie_output_projection_to_token_embeddings(token_embedding_weight):
    """Return an output projection weight that shares storage with token_embedding_weight.

    Input shape: (vocab_size, d_model). Output shape: (d_model, vocab_size).
    """
    # return an output projection weight tied to the token embedding matrix

    # In the linear layer, at position t, of (b, t, d), we take t'th row of size d and multiple it
    # by each token embedding in the vocab, and end up with a score.
    return token_embedding_weight.T

def apply_log_softmax_over_vocab(logits):
    # Convert decoder logits (B, T, V) into log probabilities over the vocabulary axis.
    # each token pos has a raw score over the entire vocab
    return torch.log_softmax(logits, dim=-1)

def run_transformer_forward(src_ids, tgt_ids, model_params, num_heads, pad_id):
    # embed src+tgt, add PE, build masks, run encoder/decoder, project to log probs.
    # token_emb -> (vocab_size, d_model)
    # output_projection -> (vocab_size, d_model)
    # Pytorch's advanced(tensor) indexing allows
     # if src_ids (2,3) and token embedding (2,4,5), then embedding[src_ids] (2,3,5)
    src_end = scale_embeddings_by_sqrt_d_model(model_params["token_embedding"][src_ids], model_params["token_embedding"].shape[-1])
    tgt_end = scale_embeddings_by_sqrt_d_model(model_params["token_embedding"][tgt_ids], model_params["token_embedding"].shape[-1])
    max_seq_length = max(src_ids.shape[-1], tgt_ids.shape[-1])
    # src_pos_encod = build_sinusoidal_positional_encoding(src_ids.shape[-1], model_params["token_embedding"].shape[-1])
    pos_encod = build_sinusoidal_positional_encoding(max_seq_length, model_params["token_embedding"].shape[-1])
    src_pos_aware_end = add_positional_encoding_to_embeddings(src_end, pos_encod)
    tgt_pos_aware_end = add_positional_encoding_to_embeddings(tgt_end, pos_encod)

    src_padding_mask = build_padding_mask(src_ids, pad_id)
    src_casual_mask = build_causal_mask(src_ids.shape[-1])
    src_padding_casual_mask = combine_padding_and_causal_masks(src_padding_mask, src_casual_mask)

    tgt_padding_mask = build_padding_mask(tgt_ids, pad_id)
    tgt_casual_mask = build_causal_mask(tgt_ids.shape[-1])
    tgt_padding_casual_mask = combine_padding_and_causal_masks(tgt_padding_mask, tgt_casual_mask)

    encoder_stack = stack_encoder_layers(src_pos_aware_end, model_params["encoder_layers"], num_heads, src_padding_casual_mask)
    decoder_stack = stack_decoder_layers(tgt_pos_aware_end, encoder_stack, model_params["decoder_layers"], num_heads, src_padding_mask, tgt_padding_casual_mask)
    final_output_projection = apply_final_output_projection(decoder_stack, model_params["output_projection"])
    softmax_prob = apply_log_softmax_over_vocab(final_output_projection)
    return softmax_prob


Parameter Initialization

In [8]:
def init_encoder_layer_parameters(d_model, num_heads, d_ff):
    """Return a dict of leaf tensors with requires_grad=True for one encoder layer."""
    # allocate w_q, w_k, w_v, w_o, w1, b1, w2, b2, attn_gamma, attn_beta, ffn_gamma, ffn_beta.
    model_params = {}
    model_params["w_q"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w_k"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w_v"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w_o"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w1"] =  (torch.randn(d_model, d_ff, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["b1"] = torch.zeros(d_ff, dtype=torch.float32, requires_grad=True)
    model_params["w2"] =  (torch.randn(d_ff, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["b2"] = torch.zeros(d_model, dtype=torch.float32, requires_grad=True)
    model_params["attn_gamma"] = torch.ones(d_model, dtype=torch.float32, requires_grad=True)
    model_params["attn_beta"] = torch.zeros(d_model, dtype=torch.float32, requires_grad=True)
    model_params["ffn_gamma"] = torch.ones(d_model, dtype=torch.float32, requires_grad=True)
    model_params["ffn_beta"] = torch.zeros(d_model, dtype=torch.float32, requires_grad=True)
    return model_params

def init_decoder_layer_parameters(d_model, num_heads, d_ff):
    # return a dict of requires_grad tensors for one decoder layer
    model_params = {}
    model_params["w_q_self"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w_q_cross"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w_k_self"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w_k_cross"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w_v_self"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w_v_cross"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w_o_self"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w_o_cross"] = (torch.randn(d_model, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["w1"] =  (torch.randn(d_model, d_ff, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["b1"] = torch.zeros(d_ff, dtype=torch.float32, requires_grad=True)
    model_params["w2"] =  (torch.randn(d_ff, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    model_params["b2"] = torch.zeros(d_model, dtype=torch.float32, requires_grad=True)
    model_params["self_gamma"] = torch.ones(d_model, dtype=torch.float32, requires_grad=True)
    model_params["self_beta"] = torch.zeros(d_model, dtype=torch.float32, requires_grad=True)
    model_params["cross_gamma"] = torch.ones(d_model, dtype=torch.float32, requires_grad=True)
    model_params["cross_beta"] = torch.zeros(d_model, dtype=torch.float32, requires_grad=True)
    model_params["ffn_gamma"] = torch.ones(d_model, dtype=torch.float32, requires_grad=True)
    model_params["ffn_beta"] = torch.zeros(d_model, dtype=torch.float32, requires_grad=True)
    return model_params

def init_embedding_and_projection_parameters(vocab_size, d_model, tie_weights=True):
    """Allocate src/tgt embeddings and output projection (optionally tied)."""
    # allocate three (vocab_size, d_model) tensors with requires_grad=True
    init_params = {}
    init_params["src_embedding"] = (torch.randn(vocab_size, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    init_params["tgt_embedding"] = (torch.randn(vocab_size, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    init_params["output_projection"] = init_params["tgt_embedding"] if tie_weights else \
    (torch.randn(vocab_size, d_model, dtype=torch.float32) * 0.1).requires_grad_()
    return init_params

def collect_model_parameters_into_list(encoder_layer_params, decoder_layer_params, embedding_params):
    # walk the encoder, decoder, and embedding dicts and return a flat deduped list of tensors
    trainable_tensors = []
    for i in range(len(encoder_layer_params)):
        trainable_tensors.extend(encoder_layer_params[i].values())
    for i in range(len(decoder_layer_params)):
        trainable_tensors.extend(decoder_layer_params[i].values())
    trainable_tensors.append(embedding_params["src_embedding"])
    trainable_tensors.append(embedding_params["tgt_embedding"])
    if embedding_params["tgt_embedding"].data_ptr() != embedding_params["output_projection"].data_ptr():
        trainable_tensors.append(embedding_params["output_projection"])
    return trainable_tensors

Training Objective and Schedule

In [55]:
def shift_targets_right_with_start_token(target_ids, start_token_id):
    # prepend start_token_id and drop the last column so output shape matches target_ids
    # teacher-forcing, decoder is fed the correct input back instead of its own output
    pad_settings = (1,0)
    return torch.nn.functional.pad(target_ids[...,:-1], pad_settings, mode='constant', value=start_token_id)

def compute_noam_learning_rate(step, d_model, warmup_steps):
    # return the Noam warmup learning rate for the given step.
    return 1/math.sqrt(d_model) * min(1/math.sqrt(step), step*1/math.sqrt(warmup_steps**3))

def build_uniform_smoothing_distribution(shape, vocab_size, epsilon):
    # return a float tensor of `shape` filled with epsilon / (vocab_size - 2).
    uniform_mass = epsilon / (vocab_size - 2)
    uniform_smoothing = torch.full(shape, uniform_mass)
    return uniform_smoothing

def set_confidence_on_gold_tokens(smoothed_distribution, gold_token_ids, confidence):
    """Place confidence mass at gold-token positions of a smoothed target distribution."""
    # write the confidence value at each gold token id along the vocab axis
    return smoothed_distribution.scatter(
            dim=-1,
            index=gold_token_ids.unsqueeze(-1),
            value=confidence
        )

def zero_pad_column_and_pad_token_rows(smoothed_distribution, gold_token_ids, pad_id):
    # zero the pad column and the rows where the gold token equals pad_id
    # col_pad, check each elem in -1 dim, row_pad, match each elem in -2 dim, if so overwrite dim -1
    smoothed_distribution[..., pad_id] = 0
    pad_row = gold_token_ids == pad_id
    smoothed_distribution[pad_row] = 0
    return smoothed_distribution

def average_loss_over_non_pad_tokens(total_loss, gold_token_ids, pad_id):
    # divide total_loss by the count of non-pad tokens in gold_token_ids
    pad_mask = gold_token_ids != pad_id
    non_pad_token = pad_mask.sum()
    return total_loss.sum() / max(non_pad_token, 1)

def compute_token_accuracy_ignoring_pad(log_probabilities, gold_token_ids, pad_id):
    # argmax over vocab, compare to gold, average over non-pad positions only
    max_across_row = torch.argmax(log_probabilities, dim=-1) # dim=-2 compares row against row
    non_pad_correct_pred_mask = max_across_row == gold_token_ids
    correct_pred_pad_check = max_across_row != pad_id
    non_pad_gold_mask = gold_token_ids != pad_id # gold and not a pad
    non_pad_correct_pred_mask = non_pad_correct_pred_mask & correct_pred_pad_check # correct pred and not a pad
    total_non_pad_correct_pred = torch.sum(non_pad_correct_pred_mask)
    total_non_pad_gold = torch.sum(non_pad_gold_mask)
    if total_non_pad_gold == 0:
      return torch.tensor(0.0)
    return total_non_pad_correct_pred / total_non_pad_gold

Adam Optimizer

In [10]:
def initialize_adam_optimizer_state(parameter_list):
    """Allocate Adam m, v zero buffers and a step counter t=0."""
    # allocate zero buffers for first and second moments, plus step counter
    if len(parameter_list) == 0:
        return {"t": 0, "m": [], "v": []}
    zero_buffer = {}
    zero_buffer["t"] = 0
    for par in parameter_list:
        zero_buffer.setdefault("m", []).append(torch.zeros_like(par))
        zero_buffer.setdefault("v", []).append(torch.zeros_like(par))

    return zero_buffer

def update_adam_first_moment(m_prev, grad, beta1):
    """Return m_t = beta1 * m_prev + (1 - beta1) * grad."""
    # apply the Adam first-moment EMA update and return the new tensor
    return beta1 * m_prev + (1 - beta1) * grad

def update_adam_second_moment(v_prev, grad, beta2):
    """Return v_t = beta2 * v_prev + (1 - beta2) * grad ** 2."""
    # apply Adam's EMA update for the second moment of the gradient
    return beta2 * v_prev + (1 - beta2) * (grad ** 2)

def apply_adam_bias_correction(m_t, v_t, beta1, beta2, step):
    """Return bias-corrected (m_hat, v_hat) for Adam at the given step."""
    # divide each moment by (1 - beta**step) using its respective beta
    m_hat = m_t / (1 - beta1 ** step)
    v_hat = v_t / (1 - beta2 ** step)
    return (m_hat, v_hat)

def apply_adam_step_to_all_parameters(parameter_list, optimizer_state, learning_rate, beta1=0.9, beta2=0.98, epsilon=1e-9):
    # increment t, then for each param with a grad update m, v, bias-correct, and subtract delta in place.
    for i in range(len(parameter_list)):
        if not parameter_list[i].requires_grad:
            continue
        optimizer_state["t"] += 1
        optimizer_state["m"][i] = update_adam_first_moment(optimizer_state["m"][i], parameter_list[i].grad, beta1)
        optimizer_state["v"][i] = update_adam_second_moment(optimizer_state["v"][i], parameter_list[i].grad, beta2)
        uncor_m, uncor_v = apply_adam_bias_correction(optimizer_state["m"][i], optimizer_state["v"][i], beta1, beta2, optimizer_state["t"])
        with torch.no_grad():
            parameter_list[i] -= (learning_rate/ (torch.sqrt(uncor_v) + epsilon)) * uncor_m
        return optimizer_state

def zero_all_parameter_gradients(parameter_list):
    """Clear the .grad of every parameter tensor before the next backward pass."""
    # clear the accumulated gradient on every parameter tensor in the list
    for param in parameter_list:
        param.grad = None



In [58]:
def run_transformer_forward(src_ids, tgt_ids, model_params, num_heads, pad_id):
    # embed src+tgt, add PE, build masks, run encoder/decoder, project to log probs.
    # token_emb -> (vocab_size, d_model)
    # output_projection -> (vocab_size, d_model)
    # Pytorch's advanced(tensor) indexing allows
     # if src_ids (2,3) and token embedding (2,4,5), then embedding[src_ids] (2,3,5)
    src_end = scale_embeddings_by_sqrt_d_model(model_params["src_embedding"][src_ids], model_params["src_embedding"].shape[-1])
    tgt_end = scale_embeddings_by_sqrt_d_model(model_params["tgt_embedding"][tgt_ids], model_params["tgt_embedding"].shape[-1])
    max_seq_length = max(src_ids.shape[-1], tgt_ids.shape[-1])
    # src_pos_encod = build_sinusoidal_positional_encoding(src_ids.shape[-1], model_params["token_embedding"].shape[-1])
    pos_encod = build_sinusoidal_positional_encoding(max_seq_length, model_params["tgt_embedding"].shape[-1])
    src_pos_aware_end = add_positional_encoding_to_embeddings(src_end, pos_encod)
    tgt_pos_aware_end = add_positional_encoding_to_embeddings(tgt_end, pos_encod)

    src_padding_mask = build_padding_mask(src_ids, pad_id)
    src_casual_mask = build_causal_mask(src_ids.shape[-1])
    src_padding_casual_mask = combine_padding_and_causal_masks(src_padding_mask, src_casual_mask)

    tgt_padding_mask = build_padding_mask(tgt_ids, pad_id)
    tgt_casual_mask = build_causal_mask(tgt_ids.shape[-1])
    tgt_padding_casual_mask = combine_padding_and_causal_masks(tgt_padding_mask, tgt_casual_mask)

    encoder_stack = stack_encoder_layers(src_pos_aware_end, model_params["encoder_layers"], num_heads, src_padding_casual_mask)
    decoder_stack = stack_decoder_layers(tgt_pos_aware_end, encoder_stack, model_params["decoder_layers"], num_heads, src_padding_mask, tgt_padding_casual_mask)
    final_output_projection = apply_final_output_projection(decoder_stack, model_params["output_projection"])
    softmax_prob = apply_log_softmax_over_vocab(final_output_projection)
    return softmax_prob

def compute_batch_training_loss(src_batch, tgt_batch, model_params, config):
    # shift targets right, run the forward pass, build smoothed targets, and average the KL loss over non-pad tokens.
    shifted_target_batch = shift_targets_right_with_start_token(tgt_batch, config["start_id"])
    output = run_transformer_forward(src_batch, shifted_target_batch, model_params, config["num_heads"], config["pad_id"])
    smoothing_distribution = build_uniform_smoothing_distribution(output.shape, config["vocab_size"], config["smoothing"])
    smoothed_target_distribution = set_confidence_on_gold_tokens(smoothing_distribution, tgt_batch, 0.9)
    zeroed_smoothed_target_distribution = zero_pad_column_and_pad_token_rows(smoothed_target_distribution, tgt_batch, config["pad_id"])
    log_output = torch.log_softmax(output, dim=-1)
    loss = -1 * zeroed_smoothed_target_distribution * log_output
    loss = loss.sum(dim=-1)
    loss = average_loss_over_non_pad_tokens(loss, tgt_batch, config["pad_id"])
    return loss

In [62]:
torch.manual_seed(3)
enc = init_encoder_layer_parameters(8, 2, 16)
dec = init_decoder_layer_parameters(8, 2, 16)
emb = init_embedding_and_projection_parameters(10, 8)
print(emb)
model_params = {'encoder_layers': [enc], 'decoder_layers': [dec], **emb}
config = {'pad_id': 0, 'start_id': 1, 'vocab_size': 10, 'smoothing': 0.1, 'd_model': 8, 'num_heads': 2}
src = torch.tensor([[2, 3, 4]])
tgt = torch.tensor([[5, 6, 7]])
loss = compute_batch_training_loss(src, tgt, model_params, config)
loss.backward()
print(model_params['src_embedding'].grad is not None, bool((model_params['src_embedding'].grad.abs().sum() > 0).item()))

{'src_embedding': tensor([[ 0.1381,  0.1401, -0.0234,  0.0361,  0.0857, -0.0924,  0.0748, -0.1089],
        [ 0.1642, -0.0470, -0.0223,  0.1461, -0.1214,  0.0223, -0.0188,  0.0666],
        [-0.0034,  0.0085, -0.2503, -0.0606,  0.0372, -0.1207, -0.0232,  0.0306],
        [ 0.0448, -0.1041, -0.0622, -0.0272,  0.1024,  0.0904, -0.1024, -0.0916],
        [-0.1142, -0.1693, -0.0427,  0.0342, -0.0554,  0.0069,  0.0311, -0.1429],
        [ 0.0112, -0.0176,  0.0380,  0.0698,  0.0893, -0.0374,  0.0469, -0.1133],
        [ 0.0261, -0.0391, -0.0402,  0.0852,  0.1557, -0.0323, -0.0761,  0.2646],
        [-0.0639, -0.0446,  0.0573,  0.0013, -0.1615, -0.0955, -0.0223, -0.0252],
        [-0.0705, -0.0516, -0.0148, -0.0840, -0.1952,  0.2973, -0.1050,  0.0939],
        [-0.1054, -0.0602,  0.0770, -0.2428, -0.0851, -0.0104,  0.0375, -0.0536]],
       requires_grad=True), 'tgt_embedding': tensor([[-0.1330, -0.1598, -0.0142,  0.1755, -0.0831,  0.2241,  0.0643,  0.0878],
        [-0.0101,  0.1631, -0.1089